# Extract and Visualize Failure Steps

This notebook extracts a specific episode, saves its video, and detects failure steps based on `temporal_disagreement`. When a failure is detected, it will fetch the current and past 5 checkpoint images to visualize and save.

In [ ]:
try:
    from tools.failure.episode_failure_extractor import EpisodeFailureExtractor
except ModuleNotFoundError:
    from episode_failure_extractor import EpisodeFailureExtractor

# === CONFIGURATION ===
episode_idx = 0
video_fps = 240
video_scale = 0.5
extractor0 = EpisodeFailureExtractor("eval/eval_pick_up_markers_cp_calibration")
extractor0.save_episode_video(episode_idx, "examples/pick_up_markers", fps=video_fps, scale=video_scale)
extractor1 = EpisodeFailureExtractor("eval/eval_pick_up_markers_failure_and_ckpt_part2")
failure_result = extractor1.save_failure_images(episode_idx, f"examples/pick_up_markers/{episode_idx}")

In [ ]:
import matplotlib.pyplot as plt

if failure_result.get("status") != "failure_found":
    print("No failed step detected in this episode.")
else:
    cameras = ["left", "middle", "right"]

    def show_three_views(images_dict, title):
        fig, axes = plt.subplots(1, len(cameras), figsize=(15, 4))
        for ax, cam in zip(axes, cameras, strict=False):
            img_path = images_dict.get(cam)
            if img_path is None:
                ax.text(0.5, 0.5, f"{cam}\n(no image)", ha="center", va="center")
                ax.set_axis_off()
                continue
            img = plt.imread(img_path)
            ax.imshow(img)
            ax.set_title(cam)
            ax.set_axis_off()
        fig.suptitle(title)
        plt.tight_layout()
        plt.show()

    print("\n=== Current failed step images ===")
    show_three_views(failure_result.get("current_images", {}), "Current failed step")

    print("\n=== Checkpoint images ===")
    for cp in failure_result.get("checkpoint_images", []):
        cp_step = cp.get("step")
        cp_offset = cp.get("offset")
        show_three_views(
            cp.get("images", {}),
            f"Checkpoint {cp_offset} (step {cp_step})",
        )